In [19]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader

load_dotenv()

loader=TextLoader("../data/text_files/ai_notes.txt")
documents=loader.load()

print(len(documents))
print(documents[0].page_content)

embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="basic_rag_learning",
)


1
Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare for diagnosis support.
Finance companies use AI for fraud detection.
Recommendation systems are powered by AI algorithms.
Natural language processing enables chatbots and assistants.
Computer vision allows machines to understand images.
AI can improve productivity through automation.
Ethical use of AI is an important topic.
Bias in training data can affect AI decisions.
Researchers continue to improve AI model efficiency.
Cloud platforms provide scalable AI infrastructure.
AI skills are becoming increasingly valuable in technology careers.


In [25]:
retreiver=vector_store.as_retriever(search_kwargs={"k":2})


question="what is ai doing?"

retrievedDocs=retreiver.invoke(question)

print(f"Question: {question}")
print("Retrieved Documents:")
# print(retrievedDocs)
# print(type(retrievedDocs[0]))

for doc in retrievedDocs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}")
    print("-----")

Question: what is ai doing?
Retrieved Documents:
Source: ../data/text_files/ai_notes.txt
Content: Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare for diagnosis support.
Finance companies use AI for fraud detection.
Recommendation systems are powered by AI algorithms.
Natural language processing enables chatbots and assistants.
Computer vision allows machines to understand images.
AI can improve productivity through automation.
Ethical use of AI is an important topic.
Bias in training data can affect AI decisions.
Researchers continue to improve AI model efficiency.
Cloud platforms provide scalable AI infrastructure.
AI skills are becoming increasingly valuable in technology careers.
-----
Source: ../data/text_files/ai_notes.txt
Content: Artificial Intelligence is transforming many industries.
AI systems can analyze larg

In [26]:
def format_retrieved_docs(retrieved_docs):
    result=''
    for doc in retrieved_docs:
        result+=f'\n\n{doc.page_content}'

    return result.strip()


formattedAnswer=format_retrieved_docs(retrievedDocs)

print("Formatted Answer:")
print(formattedAnswer)
        

Formatted Answer:
Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare for diagnosis support.
Finance companies use AI for fraud detection.
Recommendation systems are powered by AI algorithms.
Natural language processing enables chatbots and assistants.
Computer vision allows machines to understand images.
AI can improve productivity through automation.
Ethical use of AI is an important topic.
Bias in training data can affect AI decisions.
Researchers continue to improve AI model efficiency.
Cloud platforms provide scalable AI infrastructure.
AI skills are becoming increasingly valuable in technology careers.

Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare f

In [27]:
from langchain_core.runnables import RunnableLambda


formatted_docs_runnable=RunnableLambda(format_retrieved_docs)


rettiever_chain= retreiver | formatted_docs_runnable

context=rettiever_chain.invoke(question)

print(context)

Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare for diagnosis support.
Finance companies use AI for fraud detection.
Recommendation systems are powered by AI algorithms.
Natural language processing enables chatbots and assistants.
Computer vision allows machines to understand images.
AI can improve productivity through automation.
Ethical use of AI is an important topic.
Bias in training data can affect AI decisions.
Researchers continue to improve AI model efficiency.
Cloud platforms provide scalable AI infrastructure.
AI skills are becoming increasingly valuable in technology careers.

Artificial Intelligence is transforming many industries.
AI systems can analyze large amounts of data.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
AI is used in healthcare for diagnosis suppo

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI


rag_prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions based on the provided context:{context}. and say no if the answer is not found in the context."),
    ("human", "{question}")
])
llm=ChatOpenAI(model="gpt-4o-mini")

rag_chain=(
    {
        'context':rettiever_chain,
        'question':RunnablePassthrough()
    }
    |rag_prompt
    |llm
    |StrOutputParser()
    
)

result=rag_chain.invoke(question)

print(result)

AI is transforming many industries by analyzing large amounts of data, supporting diagnosis in healthcare, detecting fraud in finance, powering recommendation systems, enabling chatbots and assistants through natural language processing, and allowing machines to understand images through computer vision. Additionally, AI can improve productivity through automation, and researchers continue to work on improving AI model efficiency.
